In [1]:
# pip install shared_utils

In [2]:
# Importing necessary package 
import pandas as pd 
import geopandas as gpd
import google.auth
import os
import gcsfs
import requests
import fsspec
from shapely import wkt
import re
from calitp_data_analysis.sql import get_engine
db_engine = get_engine()
credentials, project = google.auth.default()
fs = gcsfs.GCSFileSystem()

pd.set_option('display.max_columns', None)

In [3]:
GCS_FILE_PATH  = 'gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships/AHSC_2026'

In [4]:
# Load the stored ACS dataset from the specified GCS file path.
with fs.open(f"{GCS_FILE_PATH}/census_tracts_data.parquet", "rb") as f:
    tracts_ca_acs = gpd.read_parquet(f)

In [5]:
# Load the stored organization, ridership, stop, data from the specified GCS file path.
with fs.open(f"{GCS_FILE_PATH}/ridership_trips_routes_weekday.csv", "rb") as f:
    ridership_trips_routes_weekday = pd.read_csv(f)
    
with fs.open(f"{GCS_FILE_PATH}/ridership_trips_routes_saturday.csv", "rb") as f:
    ridership_trips_routes_saturday = pd.read_csv(f)
    
with fs.open(f"{GCS_FILE_PATH}/ridership_trips_routes_saturday.csv", "rb") as f:
    ridership_trips_routes_sunday = pd.read_csv(f)

In [6]:
# Load job density data from GCS and select required columns
# Open the GCS file using your existing fs object
with fs.open(f"{GCS_FILE_PATH}/job_density_2023.parquet", "rb") as f:
    jobdata = pd.read_parquet(f)

# Select only the columns you want, including geometry
jobdata = jobdata[['GEOID', 'jobs_tot']]

# Load pois data from GCS and select required columns
with fs.open(f"{GCS_FILE_PATH}/pois_2026.parquet", "rb") as f:
    pois = gpd.read_parquet(f)


## Spatial Analysis: Stop Buffers and Census Tract Intersections

In [7]:
ridership_trips_routes_weekday.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21316 entries, 0 to 21315
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   organization_name         21316 non-null  object 
 1   feed_key                  21315 non-null  object 
 2   stop_id                   21315 non-null  object 
 3   stop_name                 21316 non-null  object 
 4   stop_code                 20594 non-null  object 
 5   n_arrivals                21315 non-null  float64
 6   n_routes                  21315 non-null  float64
 7   pt_geom                   21315 non-null  object 
 8   day_type                  21316 non-null  object 
 9   route_id_list             21315 non-null  object 
 10  average_daily_boardings   21316 non-null  float64
 11  average_daily_alightings  19138 non-null  float64
 12  start_date                21316 non-null  object 
 13  end_date                  21316 non-null  object 
dtypes: flo

In [8]:
# Drop rows with missing pt_geom
ridership_trips_routes_weekday = ridership_trips_routes_weekday[
    ridership_trips_routes_weekday['pt_geom'].notna() & 
    (ridership_trips_routes_weekday['pt_geom'] != 'nan')
].copy()

ridership_trips_routes_saturday = ridership_trips_routes_saturday[
    ridership_trips_routes_saturday['pt_geom'].notna() & 
    (ridership_trips_routes_saturday['pt_geom'] != 'nan')
].copy()

ridership_trips_routes_sunday = ridership_trips_routes_sunday[
    ridership_trips_routes_sunday['pt_geom'].notna() & 
    (ridership_trips_routes_sunday['pt_geom'] != 'nan')
].copy()

In [9]:
# Ensure pt_geom is string type
ridership_trips_routes_weekday['pt_geom'] = ridership_trips_routes_weekday['pt_geom'].astype(str)
ridership_trips_routes_saturday['pt_geom'] = ridership_trips_routes_saturday['pt_geom'].astype(str)
ridership_trips_routes_sunday['pt_geom'] = ridership_trips_routes_sunday['pt_geom'].astype(str)

In [10]:
# Convert pt_geom column from WKT to shapely geometry
ridership_trips_routes_weekday['geometry'] = ridership_trips_routes_weekday['pt_geom'].apply(wkt.loads)
ridership_trips_routes_saturday['geometry'] = ridership_trips_routes_saturday['pt_geom'].apply(wkt.loads)
ridership_trips_routes_sunday['geometry'] = ridership_trips_routes_sunday['pt_geom'].apply(wkt.loads)

# Create a GeoDataFrame
gdf_ridership = gpd.GeoDataFrame(ridership_trips_routes_weekday, geometry='geometry')
gdf_ridership_saturday = gpd.GeoDataFrame(ridership_trips_routes_saturday, geometry='geometry')
gdf_ridership_sunday = gpd.GeoDataFrame(ridership_trips_routes_sunday, geometry='geometry')

In [11]:
# Set CRS (assuming WGS84)
gdf_ridership.set_crs(epsg=4326, inplace=True)
gdf_ridership_saturday.set_crs(epsg=4326, inplace=True)
gdf_ridership_sunday.set_crs(epsg=4326, inplace=True)

,organization_name,feed_key,stop_id,stop_name,stop_code,n_arrivals,n_routes,pt_geom,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date,geometry
0,Samtrans,db97cc02836aa5f0cf647d80160b23ec,345017,1000 El Camino Real-Menlo College,345017,64.0,1.0,POINT(-122.191284 37.457543),Sunday,8.800000,15.600000,2025-08-01,2025-08-31,POINT (-122.19128 37.45754)
1,Golden Gate Transit,de77cb40e92fb47fa8d16228977cfb86,40469,1011 Andersen Dr,40469,4.0,1.0,POINT(-122.504252 37.955391),Sunday,1.500000,0.000000,2025-09-01,2025-09-30,POINT (-122.50425 37.95539)
2,Long Beach Transit,cddd375786d835389a7beb9632369907,355,10th & Long Beach NW,0355,32.0,1.0,POINT(-118.189862 33.779026),Sunday,2.300620,26.405652,2024-07-01,2025-06-30,POINT (-118.18986 33.77903)
3,Long Beach Transit,cddd375786d835389a7beb9632369907,356,10th & Pine NW,0356,64.0,1.0,POINT(-118.192676 33.779068),Sunday,98.589716,76.027869,2024-07-01,2025-06-30,POINT (-118.19268 33.77907)
4,SDMTS,1fff52f9349da228c56eef492df5001b,11656,10th Av & Broadway,11656,120.0,2.0,POINT(-117.15566399 32.7159774),Sunday,46.287992,39.811145,2024-09-01,2025-01-25,POINT (-117.15566 32.71598)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12316,Caltrain,f189d5677d4a106b98585f3c5d4fd42c,70091,San Mateo,NaN,33.0,1.0,POINT(-122.323851 37.568087),Sunday,478.543508,NaN,2023-11-01,2025-07-31,POINT (-122.32385 37.56809)
12317,Caltrain,f189d5677d4a106b98585f3c5d4fd42c,70241,Santa Clara,NaN,33.0,1.0,POINT(-121.93608 37.353238),Sunday,387.793441,NaN,2023-11-01,2025-07-31,POINT (-121.93608 37.35324)
12318,Caltrain,f189d5677d4a106b98585f3c5d4fd42c,70041,South San Francisco,NaN,33.0,1.0,POINT(-122.404979051 37.655941395),Sunday,163.026362,NaN,2023-11-01,2025-07-31,POINT (-122.40498 37.65594)
12319,Caltrain,f189d5677d4a106b98585f3c5d4fd42c,70221,Sunnyvale,NaN,33.0,1.0,POINT(-122.031372 37.378916),Sunday,593.758215,NaN,2023-11-01,2025-07-31,POINT (-122.03137 37.37892)


In [12]:
# Reproject to match census tracts CRS
gdf_ridership = gdf_ridership.to_crs(tracts_ca_acs.crs)
gdf_ridership_saturday = gdf_ridership_saturday.to_crs(tracts_ca_acs.crs)
gdf_ridership_sunday = gdf_ridership_sunday.to_crs(tracts_ca_acs.crs)

In [13]:
stop_buffered = gdf_ridership.copy()
stop_buffered_saturday = gdf_ridership_saturday.copy()
stop_buffered_sunday = gdf_ridership_sunday.copy()

stop_buffered["geometry"] = stop_buffered.geometry.buffer(404.672)
stop_buffered_saturday["geometry"] = stop_buffered_saturday.geometry.buffer(404.672)
stop_buffered_sunday["geometry"] = stop_buffered_sunday.geometry.buffer(404.672)

In [14]:
# Inner join with ACS data on 'geo_id'
tracts_ca_acs = tracts_ca_acs.merge(jobdata, on = 'GEOID', how='left')

In [15]:
pois = pois.to_crs(tracts_ca_acs.crs)
pois_with_tract = gpd.sjoin(
    pois,
    tracts_ca_acs[["GEOID", "geometry"]],
    how="left",
    predicate="within"
)


In [16]:
poi_total = (
    pois_with_tract
    .groupby("GEOID")
    .size()
    .reset_index(name="poi_total")
)

In [17]:
poi_total.head(5)

,GEOID,poi_total
0,06001400100,18
1,06001400200,63
2,06001400300,95
3,06001400400,50
4,06001400500,20


In [18]:
tracts_ca_acs = tracts_ca_acs.merge(poi_total, on="GEOID", how="left")
tracts_ca_acs["poi_total"] = tracts_ca_acs["poi_total"].fillna(0)

In [19]:
tracts_ca_acs.head(5)

,STATEFP,COUNTYFP,TRACTCE,GEOIDFQ,GEOID,NAME,NAMELSAD,STUSPS,NAMELSADCO,STATE_NAME,LSAD,ALAND,AWATER,geometry,total_pop,poverty_pop,non_us_citizen,male_65_to_66,male_67_to_69,male_70_to_74,male_75_to_79,male_80_to_84,male_85_and_over,female_65_to_66,female_67_to_69,female_70_to_74,female_75_to_79,female_80_to_84,female_85_and_over,male_15_17,male_18_19,male_20,male_21,male_22_24,female_15_17,female_18_19,female_20,female_21,female_22_24,median_household_income,income_less_10000,income_10000_14999,income_15000_24999,income_25000_34999,income_35000_49999,income_50000_64999,income_65000_74999,workers_with_no_car,households_with_no_cars,public_asst_pop,veteran_pop,county_name,inc_extremelylow,inc_verylow,inc_low,inc_total_lowincome,male_seniors,female_seniors,male_youth,female_youth,total_seniors,total_youth,disabled_pop,area_m2,jobs_tot,poi_total
0,06,077,005127,1400000US06077005127,06077005127,51.27,Census Tract 51.27,CA,San Joaquin County,California,CT,1960015,0,"POLYGON ((-113121.932 -19526.254, -112924.671 ...",7077,456,489,70,17,142,90,25,12,250,206,149,63,60,0,300,138,15,37,173,148,7,54,71,128,111741,676,546,467,464,349,699,411,24,12,391,274,San Joaquin,1689,813,1110,3612,356,728,663,408,1084,1071,1008,1.962577e+06,2826.0,9.0
1,06,077,003406,1400000US06077003406,06077003406,34.06,Census Tract 34.06,CA,San Joaquin County,California,CT,839414,14789,"POLYGON ((-114750.039 2112.119, -114364.997 21...",3392,815,278,68,51,16,24,0,45,8,31,36,53,42,14,115,43,0,37,75,208,64,0,0,116,40955,257,319,252,343,194,243,49,0,126,433,84,San Joaquin,828,537,292,1657,204,184,270,388,388,658,477,8.541742e+05,1413.0,12.0
2,06,077,004402,1400000US06077004402,06077004402,44.02,Census Tract 44.02,CA,San Joaquin County,California,CT,4346359,0,"POLYGON ((-111508.542 10947.756, -111451.691 1...",6069,444,955,51,31,80,29,42,0,69,165,60,140,27,66,155,79,68,80,148,59,175,46,21,242,99338,518,259,670,479,519,425,306,28,52,149,173,San Joaquin,1447,998,731,3176,233,527,530,543,760,1073,632,4.345246e+06,2424.0,30.0
3,06,077,005108,1400000US06077005108,06077005108,51.08,Census Tract 51.08,CA,San Joaquin County,California,CT,1515789,0,"POLYGON ((-108049.919 -23086.948, -108045.413 ...",5273,1388,377,16,19,0,48,75,0,49,27,33,67,6,39,77,44,77,0,127,51,52,24,58,106,75677,445,212,336,204,647,477,132,8,50,360,103,San Joaquin,993,851,609,2453,158,221,325,291,379,616,519,1.516450e+06,2076.0,39.0
4,06,077,000401,1400000US06077000401,06077000401,4.01,Census Tract 4.01,CA,San Joaquin County,California,CT,1045658,0,"POLYGON ((-115228.788 -5005.057, -114541.480 -...",2778,656,244,7,38,37,60,5,10,45,56,27,29,51,56,99,0,0,23,150,22,0,7,0,148,63503,299,293,179,151,468,137,163,6,100,132,98,San Joaquin,771,619,300,1690,157,264,272,177,421,449,296,1.045022e+06,1477.0,14.0


In [20]:
tracts_ca_acs.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 9131 entries, 0 to 9130
Data columns (total 66 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   STATEFP                  9131 non-null   object  
 1   COUNTYFP                 9131 non-null   object  
 2   TRACTCE                  9131 non-null   object  
 3   GEOIDFQ                  9131 non-null   object  
 4   GEOID                    9131 non-null   object  
 5   NAME                     9131 non-null   object  
 6   NAMELSAD                 9131 non-null   object  
 7   STUSPS                   9131 non-null   object  
 8   NAMELSADCO               9131 non-null   object  
 9   STATE_NAME               9131 non-null   object  
 10  LSAD                     9131 non-null   object  
 11  ALAND                    9131 non-null   int64   
 12  AWATER                   9131 non-null   int64   
 13  geometry                 9131 non-null   geometry
 14  

In [21]:
tracts_ca_acs.crs

<Projected CRS: EPSG:3310>
Name: NAD83 / California Albers
Axis Info [cartesian]:
- X[east]: Easting (metre)
- Y[north]: Northing (metre)
Area of Use:
- name: United States (USA) - California.
- bounds: (-124.45, 32.53, -114.12, 42.01)
Coordinate Operation:
- name: California Albers
- method: Albers Equal Area
Datum: North American Datum 1983
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [22]:
stop_buffered.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 21315 entries, 0 to 21315
Data columns (total 15 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   organization_name         21315 non-null  object  
 1   feed_key                  21315 non-null  object  
 2   stop_id                   21315 non-null  object  
 3   stop_name                 21315 non-null  object  
 4   stop_code                 20594 non-null  object  
 5   n_arrivals                21315 non-null  float64 
 6   n_routes                  21315 non-null  float64 
 7   pt_geom                   21315 non-null  object  
 8   day_type                  21315 non-null  object  
 9   route_id_list             21315 non-null  object  
 10  average_daily_boardings   21315 non-null  float64 
 11  average_daily_alightings  19138 non-null  float64 
 12  start_date                21315 non-null  object  
 13  end_date                  21315 non-null  o

In [23]:
geometry_intersect = gpd.overlay(
    stop_buffered, 
    tracts_ca_acs, 
    how='intersection', 
    keep_geom_type=True
)


geometry_intersect_saturday = gpd.overlay(
    stop_buffered_saturday, 
    tracts_ca_acs, 
    how='intersection', 
    keep_geom_type=True
)


geometry_intersect_sunday = gpd.overlay(
    stop_buffered_sunday, 
    tracts_ca_acs, 
    how='intersection', 
    keep_geom_type=True
)

In [24]:
# Calculate intersected area
geometry_intersect['area_2'] = geometry_intersect.geometry.area
geometry_intersect_saturday['area_2'] = geometry_intersect_saturday.geometry.area
geometry_intersect_sunday['area_2'] = geometry_intersect_sunday.geometry.area

# Calculate the proportion of the tract that intersects each stop
geometry_intersect['area_ratio'] = geometry_intersect['area_2'] / geometry_intersect['area_m2']
geometry_intersect_saturday['area_ratio'] = geometry_intersect_saturday['area_2'] / geometry_intersect_saturday['area_m2']
geometry_intersect_sunday['area_ratio'] = geometry_intersect_sunday['area_2'] / geometry_intersect_sunday['area_m2']

In [25]:
# Define demographic and socioeconomic columns to be adjusted by area ratio
cols_to_weight = [
    'total_pop', 'poverty_pop', 'non_us_citizen', 'workers_with_no_car', 
    'households_with_no_cars', 'disabled_pop', 'public_asst_pop', 
    'inc_extremelylow', 'inc_verylow', 'inc_low', 
    'male_seniors', 'female_seniors', 'veteran_pop', 'male_youth', 'inc_total_lowincome',  'female_youth', 'total_seniors', 'jobs_tot', 'total_youth', 'ALAND', 'poi_total'
]

# Apply area_ratio
for col in cols_to_weight:
    geometry_intersect[f'{col}_adj'] = geometry_intersect[col] * geometry_intersect['area_ratio']

for col in cols_to_weight:
    geometry_intersect_saturday[f'{col}_adj'] = geometry_intersect_saturday[col] * geometry_intersect_saturday['area_ratio']

for col in cols_to_weight:
    geometry_intersect_sunday[f'{col}_adj'] = geometry_intersect_sunday[col] * geometry_intersect_sunday['area_ratio']

In [26]:
geometry_intersect.organization_name.unique()

array(['Gold Coast Transit', 'Samtrans', 'SDMTS', 'Fresno County',
       'SacRT Bus', 'San Francisco Bay Area Rapid Transit District',
       'Orange County Transportation Authority', 'Long Beach Transit',
       'Foothill Transit', 'Golden Gate Park Shuttle', 'Big Blue Bus',
       'Culver City Bus', 'Riverside Transit', 'Caltrain',
       'City of Burbank'], dtype=object)

In [27]:
stop_acs_rollup = geometry_intersect.groupby(
    ['feed_key', 'stop_id', 'organization_name'], 
    as_index=False
)[[f'{col}_adj' for col in cols_to_weight]].sum()

stop_acs_rollup_saturday = geometry_intersect_saturday.groupby(
    ['feed_key', 'stop_id', 'organization_name'], 
    as_index=False
)[[f'{col}_adj' for col in cols_to_weight]].sum()

stop_acs_rollup_sunday = geometry_intersect_sunday.groupby(
    ['feed_key', 'stop_id', 'organization_name'], 
    as_index=False
)[[f'{col}_adj' for col in cols_to_weight]].sum()

In [28]:
stop_route_df = gdf_ridership.merge(
    stop_acs_rollup,
    on=['feed_key', 'stop_id','organization_name'],
    how='left'
)


stop_route_df_saturday = gdf_ridership_saturday.merge(
    stop_acs_rollup_saturday,
    on=['feed_key', 'stop_id','organization_name'],
    how='left'
)

stop_route_df_sunday = gdf_ridership_sunday.merge(
    stop_acs_rollup_sunday,
    on=['feed_key', 'stop_id','organization_name'],
    how='left'
)

In [29]:
stop_route_df.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 21315 entries, 0 to 21314
Data columns (total 36 columns):
 #   Column                       Non-Null Count  Dtype   
---  ------                       --------------  -----   
 0   organization_name            21315 non-null  object  
 1   feed_key                     21315 non-null  object  
 2   stop_id                      21315 non-null  object  
 3   stop_name                    21315 non-null  object  
 4   stop_code                    20594 non-null  object  
 5   n_arrivals                   21315 non-null  float64 
 6   n_routes                     21315 non-null  float64 
 7   pt_geom                      21315 non-null  object  
 8   day_type                     21315 non-null  object  
 9   route_id_list                21315 non-null  object  
 10  average_daily_boardings      21315 non-null  float64 
 11  average_daily_alightings     19138 non-null  float64 
 12  start_date                   21315 non-null  object 

In [31]:
stop_route_df = gpd.GeoDataFrame(
    stop_route_df, 
    geometry='geometry', 
    crs=geometry_intersect.crs
)


stop_route_df_saturday = gpd.GeoDataFrame(
    stop_route_df_saturday, 
    geometry='geometry', 
    crs=geometry_intersect_saturday.crs
)

stop_route_df_sunday = gpd.GeoDataFrame(
    stop_route_df_sunday, 
    geometry='geometry', 
    crs=geometry_intersect_sunday.crs
)

In [32]:
# Store data in warehouse
with fs.open(f"{GCS_FILE_PATH}/stop_route_df.parquet", "wb") as f:
    stop_route_df.to_parquet(f, index=False)

with fs.open(f"{GCS_FILE_PATH}/stop_route_df_saturday.parquet", "wb") as f:
    stop_route_df_saturday.to_parquet(f, index=False)

with fs.open(f"{GCS_FILE_PATH}/stop_route_df_sunday.parquet", "wb") as f:
    stop_route_df_sunday.to_parquet(f, index=False)